In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 7 (optional): Visualize and sanity-check the migration labels
# =============================================================================
# Step:         7 of 7, optional / exploratory (not required for the AI-ready dataset)
# Summary:      Diagnostic plots over the Step 6 outputs: unstable-mobility time, event types, events per vehicle, spatial map.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.0.0
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 7 (optional) — Visualize and sanity-check the migration labels

**This notebook is supplementary / exploratory** - it is not required to
produce the AI-ready dataset (that is the output of Step 6). It exists to
help you sanity-check the labeling and choose a good `WINDOW_S` warning
horizon by visualizing:

1. how much of each vehicle's time is spent in "unstable" (C4) mobility,
2. the distribution of event types (normal handover / delayed handover /
   ping-pong) for one or more candidate window sizes,
3. how many handover events each vehicle experiences, and
4. where in space handover events happen, colored by destination cell.

**Inputs:** the per-window outputs of Step 6
(`dataset_labeled_w<W>.csv`, `events_all_w<W>.csv`) and
`trajectories.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# ---- Parameters ----
# Defaults compare a single window (matching Step 6's default
# WINDOW_S_VALUES=[3.0] on the bundled example) so this notebook runs out
# of the box. Re-run Step 6 with more WINDOW_S_VALUES and list them here to
# get the full side-by-side comparison this notebook is designed for.
WINDOW_S_VALUES = [3.0]   # which Step 6 outputs to compare
TOL_S = 0.1               # must match the value used in Step 6

TRAJECTORIES_PATH = "trajectories.csv"
DATASET_TEMPLATE = "dataset_labeled_w{w:g}.csv"
EVENTS_ALL_TEMPLATE = "events_all_w{w:g}.csv"

# Vehicles to exclude from the plots below as non-representative outliers
# (e.g. a vehicle that spends its entire recorded time in unstable mobility
# right at the edge of the simulation). Inspect the first boxplot, then
# fill this in and re-run if needed - starts empty.
VEHICLES_TO_EXCLUDE = []

TOP_K_BASE_STATIONS = 9   # how many destination cells get their own color on the event map
EVENT_MAP_WINDOW_S = 3.0  # which window size's events to use for the spatial map
MATCH_TOLERANCE_S = 0.05  # max time gap allowed when matching an event to a sample's (x, y)


## 1. Fraction of time each vehicle spends in unstable (C4) mobility

A vehicle's serving cell is "unstable" while it's inside a run shorter than
`WINDOW_S - TOL_S` (i.e. flickering between cells rather than settled on
one). This helper computes, per vehicle, what fraction of its total time
was spent in such runs, for one window size.


In [ ]:
def c4_fraction_per_vehicle(df, runs, thr, veh_col="veh_id", time_col="t"):
    """Fraction of each vehicle's time spent inside a short (unstable) run."""
    dt = df.groupby(veh_col)[time_col].diff().median()
    if pd.isna(dt) or dt <= 0:
        raise ValueError("Could not estimate the sampling interval dt.")

    df = df.copy()
    df["is_C4"] = False
    short_runs = runs[runs["duration"] < thr]
    for veh, grp in short_runs.groupby(veh_col):
        idx_veh = df.index[df[veh_col].eq(veh)]
        if len(idx_veh) == 0:
            continue
        t_veh = df.loc[idx_veh, time_col].to_numpy()
        for _, r in grp.iterrows():
            in_short = (t_veh >= r["start_time"]) & (t_veh <= r["end_time"])
            if in_short.any():
                df.loc[idx_veh[in_short], "is_C4"] = True

    total_time = df.groupby(veh_col).size() * dt
    c4_time = df[df["is_C4"]].groupby(veh_col).size() * dt
    return (c4_time / total_time).fillna(0)


runs = pd.read_csv(TRAJECTORIES_PATH)
w = WINDOW_S_VALUES[0]
df_w = pd.read_csv(DATASET_TEMPLATE.format(w=w))
df_w = df_w[~df_w["veh_id"].isin(VEHICLES_TO_EXCLUDE)]
runs_w = runs[~runs["veh_id"].isin(VEHICLES_TO_EXCLUDE)]

frac_c4 = c4_fraction_per_vehicle(df_w, runs_w, thr=w - TOL_S)
print(f"window_s={w}")
print("mean % time in C4:", 100 * frac_c4.mean())
print("median % time in C4:", 100 * frac_c4.median())
print("p90 % time in C4:", 100 * frac_c4.quantile(0.90))
print("\nvehicles spending 100% of their time in C4 (candidates for VEHICLES_TO_EXCLUDE):")
print(frac_c4[frac_c4 == 1.0])

plt.figure()
plt.boxplot(frac_c4.values, showfliers=True)
plt.ylabel("Fraction of time in C4 (unstable)")
plt.title(f"Time spent in unstable mobility per vehicle (window_s={w})")
plt.show()


## 2. Event-type distribution, compared across window sizes

`C4_no_estable` is excluded here since it is a diagnostic state (entering a
short run), not one of the four handover outcome types.


In [ ]:
# fixed left-to-right order/labels so the bars line up the same way across
# every subplot, regardless of which case happens to be most frequent
EVENT_ORDER = ["C1_handover_normal", "C2a_ABC", "C2b_handover_sin_historico", "C3_pingpong"]
EVENT_LABELS = ["C1: normal handover", "C2a: delayed (A-B-C)", "C2b: no prior history", "C3: ping-pong (A-B-A)"]
SHORT_LABELS = ["C1", "C2a", "C2b", "C3"]

# one subplot per window size, side by side, sharing the y-axis so bar
# heights are directly comparable across window sizes
fig, axes = plt.subplots(1, len(WINDOW_S_VALUES), figsize=(4 * len(WINDOW_S_VALUES), 4), sharey=True)
if len(WINDOW_S_VALUES) == 1:
    axes = [axes]  # plt.subplots returns a bare Axes (not an array) when there is only one

for ax, w in zip(axes, WINDOW_S_VALUES):
    ev = pd.read_csv(EVENTS_ALL_TEMPLATE.format(w=w))
    ev = ev[ev["veh_id"].isin(VEHICLES_TO_EXCLUDE) == False]
    ev_f = ev[ev["case"] != "C4_no_estable"]
    counts = ev_f["case"].value_counts().reindex(EVENT_ORDER, fill_value=0)
    pct = 100 * counts / counts.sum() if counts.sum() else counts
    ax.bar(SHORT_LABELS, pct.values, color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
    ax.set_title(f"window_s={w}")
    ax.set_xlabel("Event type")

axes[0].set_ylabel("Percentage of events (%)")
fig.legend(SHORT_LABELS, labels=EVENT_LABELS, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
plt.tight_layout()
plt.show()


## 3. Number of handover events per vehicle

In [ ]:
w = EVENT_MAP_WINDOW_S
ev = pd.read_csv(EVENTS_ALL_TEMPLATE.format(w=w))
ev = ev[~ev["veh_id"].isin(VEHICLES_TO_EXCLUDE)]
ev_f = ev[ev["case"] != "C4_no_estable"]

events_per_vehicle = ev_f.groupby("veh_id").size()
veh_count_per_n_events = events_per_vehicle.value_counts().sort_index()
pct_veh = 100 * veh_count_per_n_events / veh_count_per_n_events.sum()

plt.figure()
plt.bar(pct_veh.index.astype(int), pct_veh.values)
plt.xlabel("Number of events per vehicle")
plt.ylabel("Percentage of vehicles (%)")
plt.title(f"Events per vehicle (window_s={w})")
plt.xticks(pct_veh.index.astype(int))
plt.tight_layout()
plt.show()

pct_veh.round(2)


## 4. Spatial map of handover events, colored by destination cell

Matches each event to the nearest-in-time sample of the same vehicle (within
`MATCH_TOLERANCE_S`) to recover its `(x, y)` position.


In [ ]:
# the labeled dataset (one row per vehicle per sample) - source of (x, y) positions
dataset = pd.read_csv(DATASET_TEMPLATE.format(w=EVENT_MAP_WINDOW_S))
dataset = dataset.dropna(subset=["veh_id", "t", "x", "y"]).copy()
dataset["veh_id"] = dataset["veh_id"].round().astype(int)
dataset = dataset.sort_values(["veh_id", "t"], kind="mergesort").reset_index(drop=True)

# the event log (one row per detected handover) - has no (x, y) of its own
events = ev.dropna(subset=["veh_id", "t_change"]).copy()
events["veh_id"] = events["veh_id"].round().astype(int)
events = events[events["case"] != "C4_no_estable"]

# per-vehicle sorted (time, x, y) arrays, so we can binary-search for the
# sample closest in time to each event instead of scanning the whole dataset
sample_index = {
    vid: (g["t"].to_numpy(), g["x"].to_numpy(), g["y"].to_numpy())
    for vid, g in dataset.groupby("veh_id", sort=False)
}

# for each event, find the same vehicle's dataset row with the closest
# timestamp (checking the insertion point and the one just before it is
# enough since the array is sorted) and borrow its (x, y)
xs, ys, matched = [], [], []
for row in events.itertuples(index=False):
    vid, t_change = row.veh_id, row.t_change
    if vid not in sample_index:
        xs.append(np.nan); ys.append(np.nan); matched.append(False)
        continue
    t_arr, x_arr, y_arr = sample_index[vid]
    pos = np.searchsorted(t_arr, t_change)
    candidates = [i for i in (pos, pos - 1) if 0 <= i < len(t_arr)]
    best_i = min(candidates, key=lambda i: abs(t_arr[i] - t_change))
    if abs(t_arr[best_i] - t_change) <= MATCH_TOLERANCE_S:
        xs.append(x_arr[best_i]); ys.append(y_arr[best_i]); matched.append(True)
    else:
        xs.append(np.nan); ys.append(np.nan); matched.append(False)  # no sample close enough in time

events["x_event"], events["y_event"], events["matched"] = xs, ys, matched
print(f"{100 * (1 - events['matched'].mean()):.1f}% of events could not be matched to a position within tolerance")
events = events[events["matched"]].copy()

# give each of the TOP_K_BASE_STATIONS most common destination cells its own
# color; group everything else under "other" so the legend stays readable
top_cells = events["to_cell"].value_counts().head(TOP_K_BASE_STATIONS).index.tolist()
events["cell_group"] = np.where(events["to_cell"].isin(top_cells),
                                 events["to_cell"].astype(int).astype(str), "other")
group_order = sorted((g for g in events["cell_group"].unique() if g != "other"), key=int)
if "other" in events["cell_group"].unique():
    group_order.append("other")

plt.figure(figsize=(7, 7))
for g in group_order:
    sub = events[events["cell_group"] == g]
    plt.scatter(sub["x_event"], sub["y_event"], s=12, alpha=0.6,
                label=f"cell {g}" if g != "other" else "other")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Handover event locations, colored by destination cell (window_s={EVENT_MAP_WINDOW_S})")
plt.legend(loc="best", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

events["to_cell"].value_counts(dropna=True).rename_axis("to_cell").reset_index(name="n_events")
